In [62]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split

In [63]:
df = pd.DataFrame({
    "soil_moisture": [0.10, 0.15, 0.20, 0.25, 0.40, 0.60, 0.35, 0.18,
                      0.45, 0.05, 0.80, 0.27, 0.55, 0.70, 0.12, 0.30],
    "temperature_c": [34, 30, 26, 22, 28, 30, 19, 22,
                      35, 24, 33, 33, 21, 25, 20, 29],
    "sunlight_hours": [9, 8, 7, 4, 8, 10, 3, 10,
                       12, 5, 9, 11, 2, 6, 1, 9],
    "needs_water": [1, 1, 1, 0, 0, 0, 0, 1,
                    0, 1, 0, 1, 0, 0, 1, 1]
})


In [64]:
df

,soil_moisture,temperature_c,sunlight_hours,needs_water
0,0.10,34,9,1
1,0.15,30,8,1
2,0.20,26,7,1
3,0.25,22,4,0
4,0.40,28,8,0
5,0.60,30,10,0
6,0.35,19,3,0
7,0.18,22,10,1
8,0.45,35,12,0
9,0.05,24,5,1


In [65]:
X = df[['soil_moisture', 'temperature_c', 'sunlight_hours']]
y = df['needs_water']


In [66]:
X_min = X.min()
X_max = X.max()
X_scaled = (X - X_min) / (X_max - X_min + 1e-8)

In [67]:
X_scaled

,soil_moisture,temperature_c,sunlight_hours
0,0.066667,0.9375,0.727273
1,0.133333,0.6875,0.636364
2,0.200000,0.4375,0.545455
3,0.266667,0.1875,0.272727
4,0.466667,0.5625,0.636364
5,0.733333,0.6875,0.818182
6,0.400000,0.0000,0.181818
7,0.173333,0.1875,0.818182
8,0.533333,1.0000,1.000000
9,0.000000,0.3125,0.363636


In [68]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

In [69]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

In [70]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [71]:
history = model.fit(
    X_train.values,y_train.values,
    validation_data = (X_test.values,y_test.values),
    epochs = 100,batch_size = 1 , verbose = 1
)


Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.5833 - loss: 0.6929 - val_accuracy: 0.7500 - val_loss: 0.6387
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5833 - loss: 0.6884 - val_accuracy: 0.7500 - val_loss: 0.6355
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5833 - loss: 0.6859 - val_accuracy: 0.7500 - val_loss: 0.6330
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5833 - loss: 0.6828 - val_accuracy: 0.7500 - val_loss: 0.6304
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7500 - loss: 0.6785 - val_accuracy: 0.7500 - val_loss: 0.6277
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7500 - loss: 0.6756 - val_accuracy: 0.7500 - val_loss: 0.6252
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7500 - loss: 0.6727 - val_accuracy: 0.7500 - val_loss: 0.6224
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8333 - loss: 0.6686 - val_accuracy: 0.7500 - 

In [72]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [73]:
X_train_tensor = torch.tensor(
    X_train.values, dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test.values, dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values, dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test.values, dtype=torch.float32
).reshape(-1, 1)


In [74]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True
)


In [75]:
class Model(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


model = Model(X_train.shape[1])


In [76]:
criterion = nn.BCELoss()


In [77]:
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)


In [78]:
epochs = 100

for epoch in range(epochs):

    model.train()

    for X_batch, y_batch in train_loader:

        # Forward pass
        predictions = model(X_batch)

        # Calculate loss
        loss = criterion(predictions, y_batch)

        # Clear previous gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()


In [79]:
    model.eval()

    with torch.no_grad():

        test_predictions = model(X_test_tensor)

        val_loss = criterion(
            test_predictions,
            y_test_tensor
        )

        predicted_classes = (
            test_predictions >= 0.5
        ).float()

        accuracy = (
            predicted_classes == y_test_tensor
        ).float().mean()


    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {loss.item():.4f} "
        f"Val Loss: {val_loss.item():.4f} "
        f"Val Accuracy: {accuracy.item():.4f}"
    )

Epoch [100/100] Loss: 0.5060 Val Loss: 0.5700 Val Accuracy: 1.0000


In [80]:
from tensorflow.keras import optimizers

In [81]:
opt = optimizers.SGD(learning_rate = 0.01, momentum = 0.9)